<a href="https://colab.research.google.com/github/Tipusultan199/kkk/blob/main/Phase1B_Colab_TrainSchema_Debug_READY.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 1B — Training Schema Pruning / Verification Debug Notebook

This notebook keeps the main Phase 1B logic from your original notebook, but organizes it cell by cell for Colab.

Goal:
- Load one Phase 1A-cleaned CSV.
- Verify metadata and required signal columns.
- Drop non-training / logging / 3D geometry columns.
- Enforce one canonical training schema.
- Coerce numeric columns.
- Export the cleaned Phase 1B CSV, schema lock JSON, and prune report.
- Print `✅ CLEANING APPLIED CORRECTLY` or warnings after each important step.

In [2]:
# ============================================================
# CELL 1 — Import libraries and define verification print helper
# This cell loads required packages and defines PASS/FAIL printing.
# ============================================================

from __future__ import annotations
from pathlib import Path
import json
import hashlib
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)

def print_step_result(step_name: str, passed: bool):
    if passed:
        print(f"✅ CLEANING APPLIED CORRECTLY — {step_name}")
    else:
        print(f"❌ CLEANING NOT APPLIED CORRECTLY — {step_name}")

In [3]:
# ============================================================
# CELL 2 — Upload/select the Phase 1A-cleaned CSV file
# This cell lets you upload one CSV to Colab and sets output paths.
# ============================================================

USE_COLAB_UPLOAD = True

OUTPUT_DIR = Path("phase1B_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if USE_COLAB_UPLOAD:
    try:
        from google.colab import files
        uploaded = files.upload()
        if len(uploaded) == 0:
            raise ValueError("No file uploaded.")
        INPUT_CSV = Path(list(uploaded.keys())[0])
    except Exception as e:
        raise RuntimeError(
            "Colab upload failed. If you are not in Colab, set USE_COLAB_UPLOAD=False "
            "and manually set INPUT_CSV = Path('your_file.csv')."
        ) from e
else:
    # Use this line when running outside Colab.
    INPUT_CSV = Path("063_T24.csv")

print("Input CSV:", INPUT_CSV)
print("Input exists:", INPUT_CSV.exists())
print("Output directory:", OUTPUT_DIR)

print_step_result("CELL 2 — File selected", INPUT_CSV.exists())

Saving 063_T24.csv to 063_T24 (1).csv
Input CSV: 063_T24 (1).csv
Input exists: True
Output directory: phase1B_outputs
✅ CLEANING APPLIED CORRECTLY — CELL 2 — File selected


In [4]:
# ============================================================
# CELL 3 — Define the original Phase 1B schema settings
# This cell keeps the same canonical columns, drop list, and defaults.
# ============================================================

# Real vs MI: require EMG only for real trials
REQUIRE_EMG = True

# Keep for later metadata
META_ONLY = ["subject_id", "task", "trial", "Timestamp_seconds"]

# Core inputs for training
EEG_COLS = [f"Ch{i}" for i in range(1, 9)]  # EEG channels
EMG_RAW  = [f"Ch{i} EMG raw" for i in range(1, 5)]

ET_CORE = [
    "ET_GazeLeftx","ET_GazeLefty","ET_GazeRightx","ET_GazeRighty",
    "ET_PupilLeft","ET_PupilRight",
    "ET_ValidityLeftEye","ET_ValidityRightEye",
    "ET_Blink","ET_Fixation","ET_Worn",
]

IMU_HEAD = [
    "ET_GyroX","ET_GyroY","ET_GyroZ","ET_AccX","ET_AccY","ET_AccZ",
    "ET_HeadRotationPitch","ET_HeadRotationYaw","ET_HeadRotationRoll"
]

ET_DIST = ["ET_DistanceLeft","ET_DistanceRight"]

# Always drop: logging/timekeeping/3D eye geometry/stray event columns
ALWAYS_DROP = {
    "Row","Timestamp","SampleNumber","ET_TimeSignal","LSL Timestamp",
    "EventSource","SlideEvent","StimType","Duration","CollectionPhase","SourceStimuliName",
    "EventSource.1","EventSource.2","EventSource.3",
    "ET_CameraLeftX","ET_CameraLeftY","ET_CameraRightX","ET_CameraRightY",
    "ET_Gaze3dEyeballXLeft","ET_Gaze3dEyeballYLeft","ET_Gaze3dEyeballZLeft",
    "ET_Gaze3dEyeballXRight","ET_Gaze3dEyeballYRight","ET_Gaze3dEyeballZRight",
    "ET_Gaze3dOpticalAxisXLeft","ET_Gaze3dOpticalAxisYLeft","ET_Gaze3dOpticalAxisZLeft",
    "ET_Gaze3dOpticalAxisXRight","ET_Gaze3dOpticalAxisYRight","ET_Gaze3dOpticalAxisZRight",
    "ET_Gaze3dEyelidAngleTopLeft","ET_Gaze3dEyelidAngleBottomLeft",
    "ET_Gaze3dEyelidAngleTopRight","ET_Gaze3dEyelidAngleBottomRight",
    "ET_Gaze3dEyelidApertureLeft","ET_Gaze3dEyelidApertureRight",
}

# Canonical training schema
CANONICAL_ORDER = (
    META_ONLY
    + EEG_COLS
    + EMG_RAW
    + ET_CORE
    + ET_DIST
    + IMU_HEAD
)

REQUIRED_MIN = set(META_ONLY + EEG_COLS + ET_CORE + IMU_HEAD + ET_DIST)
if REQUIRE_EMG:
    REQUIRED_MIN |= set(EMG_RAW)

# Default fillers for missing columns
FILL_DEFAULTS = {
    "ET_DistanceLeft": -1.0,
    "ET_DistanceRight": -1.0,
    "ET_Blink": 0,
    "ET_Fixation": 0,
    "ET_Worn": 1,
    "ET_ValidityLeftEye": 0,
    "ET_ValidityRightEye": 0,
}

NUMERIC_LIKE = set(CANONICAL_ORDER) - set(META_ONLY)

print("Canonical schema column count:", len(CANONICAL_ORDER))
print("Required minimum column count:", len(REQUIRED_MIN))
print("REQUIRE_EMG:", REQUIRE_EMG)
print("First 10 canonical columns:", CANONICAL_ORDER[:10])

passed = len(CANONICAL_ORDER) == 38 and all(c in CANONICAL_ORDER for c in META_ONLY)
print_step_result("CELL 3 — Schema settings loaded", passed)

Canonical schema column count: 38
Required minimum column count: 38
REQUIRE_EMG: True
First 10 canonical columns: ['subject_id', 'task', 'trial', 'Timestamp_seconds', 'Ch1', 'Ch2', 'Ch3', 'Ch4', 'Ch5', 'Ch6']
✅ CLEANING APPLIED CORRECTLY — CELL 3 — Schema settings loaded


In [5]:
# ============================================================
# CELL 4 — Load CSV safely and inspect raw input
# This cell reads the uploaded Phase 1A-cleaned CSV and shows shape/columns.
# ============================================================

def read_csv_safely(path: Path) -> pd.DataFrame | None:
    try:
        return pd.read_csv(path, encoding="utf-8-sig", on_bad_lines="skip")
    except Exception:
        try:
            return pd.read_csv(path, encoding="utf-8-sig", engine="python", on_bad_lines="skip")
        except Exception:
            return None

df_raw = read_csv_safely(INPUT_CSV)

if df_raw is None:
    raise RuntimeError(f"Could not read CSV: {INPUT_CSV}")

print("Raw shape:", df_raw.shape)
print("Raw column count:", len(df_raw.columns))
print("\nRaw columns:")
print(list(df_raw.columns))

print("\nRaw preview:")
display(df_raw.head(5))

passed = df_raw is not None and df_raw.shape[0] > 0 and df_raw.shape[1] > 0
print_step_result("CELL 4 — Raw CSV loaded", passed)

Raw shape: (11902, 55)
Raw column count: 55

Raw columns:
['subject_id', 'task', 'trial', 'Timestamp_seconds', 'Row', 'Timestamp', 'SampleNumber', 'Ch1 EMG raw', 'Ch2 EMG raw', 'Ch3 EMG raw', 'Ch4 EMG raw', 'ET_TimeSignal', 'ET_PupilLeft', 'ET_PupilRight', 'ET_DistanceLeft', 'ET_DistanceRight', 'ET_GazeLeftx', 'ET_GazeLefty', 'ET_GazeRightx', 'ET_GazeRighty', 'ET_ValidityLeftEye', 'ET_ValidityRightEye', 'ET_GyroX', 'ET_GyroY', 'ET_GyroZ', 'ET_AccX', 'ET_AccY', 'ET_AccZ', 'ET_HeadRotationPitch', 'ET_HeadRotationYaw', 'ET_HeadRotationRoll', 'ET_Blink', 'ET_Gaze3dEyeballXLeft', 'ET_Gaze3dEyeballYLeft', 'ET_Gaze3dEyeballZLeft', 'ET_Gaze3dEyeballXRight', 'ET_Gaze3dEyeballYRight', 'ET_Gaze3dEyeballZRight', 'ET_Gaze3dOpticalAxisXLeft', 'ET_Gaze3dOpticalAxisYLeft', 'ET_Gaze3dOpticalAxisZLeft', 'ET_Gaze3dOpticalAxisXRight', 'ET_Gaze3dOpticalAxisYRight', 'ET_Gaze3dOpticalAxisZRight', 'ET_Fixation', 'ET_Worn', 'LSL Timestamp', 'Ch1', 'Ch2', 'Ch3', 'Ch4', 'Ch5', 'Ch6', 'Ch7', 'Ch8']

Raw preview:


,subject_id,task,trial,Timestamp_seconds,Row,Timestamp,SampleNumber,Ch1 EMG raw,Ch2 EMG raw,Ch3 EMG raw,Ch4 EMG raw,ET_TimeSignal,ET_PupilLeft,ET_PupilRight,ET_DistanceLeft,ET_DistanceRight,ET_GazeLeftx,ET_GazeLefty,ET_GazeRightx,ET_GazeRighty,ET_ValidityLeftEye,ET_ValidityRightEye,ET_GyroX,ET_GyroY,ET_GyroZ,ET_AccX,ET_AccY,ET_AccZ,ET_HeadRotationPitch,ET_HeadRotationYaw,ET_HeadRotationRoll,ET_Blink,ET_Gaze3dEyeballXLeft,ET_Gaze3dEyeballYLeft,ET_Gaze3dEyeballZLeft,ET_Gaze3dEyeballXRight,ET_Gaze3dEyeballYRight,ET_Gaze3dEyeballZRight,ET_Gaze3dOpticalAxisXLeft,ET_Gaze3dOpticalAxisYLeft,ET_Gaze3dOpticalAxisZLeft,ET_Gaze3dOpticalAxisXRight,ET_Gaze3dOpticalAxisYRight,ET_Gaze3dOpticalAxisZRight,ET_Fixation,ET_Worn,LSL Timestamp,Ch1,Ch2,Ch3,Ch4,Ch5,Ch6,Ch7,Ch8
0,10,2,4,0.000000e+00,3,8.42220,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,535400.676423,104424.920,97475.305,104743.68,117912.734,134634.22,61569.080,62151.535,45519.785
1,10,2,4,1.499429e-07,4,8.42235,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,535400.676424,104424.920,97475.305,104743.68,117912.734,134634.22,61569.080,62151.535,45519.785
2,10,2,4,2.000000e-03,5,10.42220,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,535400.678423,104432.734,97476.400,104738.30,117938.625,134648.58,61575.230,62170.227,45518.310
3,10,2,4,2.000150e-03,6,10.42235,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,535400.678424,104432.734,97476.400,104738.30,117938.625,134648.58,61575.230,62170.227,45518.310
4,10,2,4,4.000000e-03,7,12.42220,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,535400.680423,104458.960,97479.930,104761.28,117996.180,134697.36,61586.434,62254.438,45564.180


✅ CLEANING APPLIED CORRECTLY — CELL 4 — Raw CSV loaded


In [6]:
# ============================================================
# CELL 5 — Check metadata and required columns before pruning
# This cell verifies subject_id, task, trial, Timestamp_seconds, EEG, EMG, and ET columns.
# ============================================================

existing_cols = set(df_raw.columns)

missing_canonical_before = [c for c in CANONICAL_ORDER if c not in existing_cols]
missing_required_before = sorted([c for c in REQUIRED_MIN if c not in existing_cols])
extra_cols_before = [c for c in df_raw.columns if c not in CANONICAL_ORDER]

print("Missing canonical columns before pruning:", missing_canonical_before)
print("Missing required columns before pruning :", missing_required_before)
print("Extra columns before pruning count      :", len(extra_cols_before))
print("Extra columns before pruning:")
print(extra_cols_before)

print("\nMetadata unique values:")
for c in META_ONLY[:3]:
    if c in df_raw.columns:
        print(f"{c}:", sorted(pd.Series(df_raw[c]).dropna().unique())[:20])
    else:
        print(f"{c}: MISSING")

if "Timestamp_seconds" in df_raw.columns:
    print("\nTimestamp_seconds range:")
    print("min:", df_raw["Timestamp_seconds"].min())
    print("max:", df_raw["Timestamp_seconds"].max())
    print("monotonic increasing:", df_raw["Timestamp_seconds"].is_monotonic_increasing)

passed = len(missing_required_before) == 0
if not passed:
    print("WARNING: Some required columns are missing before pruning. If DELETE_POLICY='add_missing', missing canonical columns can be added with defaults/NaN.")

print_step_result("CELL 5 — Required-column precheck", passed)

Missing canonical columns before pruning: []
Missing required columns before pruning : []
Extra columns before pruning count      : 17
Extra columns before pruning:
['Row', 'Timestamp', 'SampleNumber', 'ET_TimeSignal', 'ET_Gaze3dEyeballXLeft', 'ET_Gaze3dEyeballYLeft', 'ET_Gaze3dEyeballZLeft', 'ET_Gaze3dEyeballXRight', 'ET_Gaze3dEyeballYRight', 'ET_Gaze3dEyeballZRight', 'ET_Gaze3dOpticalAxisXLeft', 'ET_Gaze3dOpticalAxisYLeft', 'ET_Gaze3dOpticalAxisZLeft', 'ET_Gaze3dOpticalAxisXRight', 'ET_Gaze3dOpticalAxisYRight', 'ET_Gaze3dOpticalAxisZRight', 'LSL Timestamp']

Metadata unique values:
subject_id: [np.int64(10)]
task: [np.int64(2)]
trial: [np.int64(4)]

Timestamp_seconds range:
min: 0.0
max: 8.390496599999999
monotonic increasing: True
✅ CLEANING APPLIED CORRECTLY — CELL 5 — Required-column precheck


In [7]:
# ============================================================
# CELL 6 — Add missing canonical columns if needed
# This cell keeps the original add_missing policy and shows what was added.
# ============================================================

DELETE_POLICY = "add_missing"

def add_missing_columns(df: pd.DataFrame, missing_cols: list) -> pd.DataFrame:
    for c in missing_cols:
        default = FILL_DEFAULTS.get(c, np.nan)
        df[c] = default
    return df

df_work = df_raw.copy()
cols_present = set(df_work.columns)
missing_canonical = [c for c in CANONICAL_ORDER if c not in cols_present]

added_missing = []
if DELETE_POLICY == "add_missing" and missing_canonical:
    df_work = add_missing_columns(df_work, missing_canonical)
    added_missing = missing_canonical

print("DELETE_POLICY:", DELETE_POLICY)
print("Columns added:", added_missing if added_missing else "None")
print("Shape before adding missing:", df_raw.shape)
print("Shape after adding missing :", df_work.shape)

missing_required_after_add = sorted([c for c in REQUIRED_MIN if c not in set(df_work.columns)])
print("Missing required after add:", missing_required_after_add)

passed = len(missing_required_after_add) == 0
print_step_result("CELL 6 — Missing canonical columns handled", passed)

DELETE_POLICY: add_missing
Columns added: None
Shape before adding missing: (11902, 55)
Shape after adding missing : (11902, 55)
Missing required after add: []
✅ CLEANING APPLIED CORRECTLY — CELL 6 — Missing canonical columns handled


In [8]:
# ============================================================
# CELL 7 — Drop logging/timekeeping/3D geometry and non-training columns
# This cell removes columns not included in the canonical training schema.
# ============================================================

cols_present = set(df_work.columns)
keep_set = set(CANONICAL_ORDER)

drop_set = (cols_present - keep_set) | (cols_present & ALWAYS_DROP)
drop_cols = [c for c in drop_set if c in df_work.columns]

df_dropped = df_work.drop(columns=drop_cols, errors="ignore")

print("Columns dropped count:", len(drop_cols))
print("Columns dropped:")
print(sorted(drop_cols))

print("\nShape before drop:", df_work.shape)
print("Shape after drop :", df_dropped.shape)

remaining_extra = [c for c in df_dropped.columns if c not in CANONICAL_ORDER]
print("Remaining extra columns after drop:", remaining_extra)

passed = len(remaining_extra) == 0
print_step_result("CELL 7 — Non-training columns dropped", passed)

Columns dropped count: 17
Columns dropped:
['ET_Gaze3dEyeballXLeft', 'ET_Gaze3dEyeballXRight', 'ET_Gaze3dEyeballYLeft', 'ET_Gaze3dEyeballYRight', 'ET_Gaze3dEyeballZLeft', 'ET_Gaze3dEyeballZRight', 'ET_Gaze3dOpticalAxisXLeft', 'ET_Gaze3dOpticalAxisXRight', 'ET_Gaze3dOpticalAxisYLeft', 'ET_Gaze3dOpticalAxisYRight', 'ET_Gaze3dOpticalAxisZLeft', 'ET_Gaze3dOpticalAxisZRight', 'ET_TimeSignal', 'LSL Timestamp', 'Row', 'SampleNumber', 'Timestamp']

Shape before drop: (11902, 55)
Shape after drop : (11902, 38)
Remaining extra columns after drop: []
✅ CLEANING APPLIED CORRECTLY — CELL 7 — Non-training columns dropped


In [9]:
# ============================================================
# CELL 8 — Coerce numeric columns and enforce canonical order
# This cell converts signal columns to numeric and reorders columns exactly.
# ============================================================

def coerce_types(df: pd.DataFrame) -> pd.DataFrame:
    for c in df.columns:
        if c in NUMERIC_LIKE:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

df_pruned = df_dropped.copy()
df_pruned = coerce_types(df_pruned)
df_pruned = df_pruned.reindex(columns=CANONICAL_ORDER, fill_value=np.nan)

schema_sig = hashlib.md5(("|".join(df_pruned.columns)).encode()).hexdigest()[:8]

print("Final shape:", df_pruned.shape)
print("Final schema signature:", schema_sig)
print("\nFinal columns:")
print(list(df_pruned.columns))

print("\nFinal preview:")
display(df_pruned.head(5))

numeric_type_check = {
    c: str(df_pruned[c].dtype)
    for c in df_pruned.columns
    if c in NUMERIC_LIKE
}
print("\nNumeric-like column dtypes:")
print(numeric_type_check)

passed = list(df_pruned.columns) == CANONICAL_ORDER and df_pruned.shape[1] == len(CANONICAL_ORDER)
print_step_result("CELL 8 — Numeric coercion and canonical order", passed)

Final shape: (11902, 38)
Final schema signature: b9f372cf

Final columns:
['subject_id', 'task', 'trial', 'Timestamp_seconds', 'Ch1', 'Ch2', 'Ch3', 'Ch4', 'Ch5', 'Ch6', 'Ch7', 'Ch8', 'Ch1 EMG raw', 'Ch2 EMG raw', 'Ch3 EMG raw', 'Ch4 EMG raw', 'ET_GazeLeftx', 'ET_GazeLefty', 'ET_GazeRightx', 'ET_GazeRighty', 'ET_PupilLeft', 'ET_PupilRight', 'ET_ValidityLeftEye', 'ET_ValidityRightEye', 'ET_Blink', 'ET_Fixation', 'ET_Worn', 'ET_DistanceLeft', 'ET_DistanceRight', 'ET_GyroX', 'ET_GyroY', 'ET_GyroZ', 'ET_AccX', 'ET_AccY', 'ET_AccZ', 'ET_HeadRotationPitch', 'ET_HeadRotationYaw', 'ET_HeadRotationRoll']

Final preview:


,subject_id,task,trial,Timestamp_seconds,Ch1,Ch2,Ch3,Ch4,Ch5,Ch6,Ch7,Ch8,Ch1 EMG raw,Ch2 EMG raw,Ch3 EMG raw,Ch4 EMG raw,ET_GazeLeftx,ET_GazeLefty,ET_GazeRightx,ET_GazeRighty,ET_PupilLeft,ET_PupilRight,ET_ValidityLeftEye,ET_ValidityRightEye,ET_Blink,ET_Fixation,ET_Worn,ET_DistanceLeft,ET_DistanceRight,ET_GyroX,ET_GyroY,ET_GyroZ,ET_AccX,ET_AccY,ET_AccZ,ET_HeadRotationPitch,ET_HeadRotationYaw,ET_HeadRotationRoll
0,10,2,4,0.000000e+00,104424.920,97475.305,104743.68,117912.734,134634.22,61569.080,62151.535,45519.785,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10,2,4,1.499429e-07,104424.920,97475.305,104743.68,117912.734,134634.22,61569.080,62151.535,45519.785,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10,2,4,2.000000e-03,104432.734,97476.400,104738.30,117938.625,134648.58,61575.230,62170.227,45518.310,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10,2,4,2.000150e-03,104432.734,97476.400,104738.30,117938.625,134648.58,61575.230,62170.227,45518.310,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10,2,4,4.000000e-03,104458.960,97479.930,104761.28,117996.180,134697.36,61586.434,62254.438,45564.180,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Numeric-like column dtypes:
{'Ch1': 'float64', 'Ch2': 'float64', 'Ch3': 'float64', 'Ch4': 'float64', 'Ch5': 'float64', 'Ch6': 'float64', 'Ch7': 'float64', 'Ch8': 'float64', 'Ch1 EMG raw': 'float64', 'Ch2 EMG raw': 'float64', 'Ch3 EMG raw': 'float64', 'Ch4 EMG raw': 'float64', 'ET_GazeLeftx': 'float64', 'ET_GazeLefty': 'float64', 'ET_GazeRightx': 'float64', 'ET_GazeRighty': 'float64', 'ET_PupilLeft': 'float64', 'ET_PupilRight': 'float64', 'ET_ValidityLeftEye': 'int64', 'ET_ValidityRightEye': 'int64', 'ET_Blink': 'int64', 'ET_Fixation': 'int64', 'ET_Worn': 'int64', 'ET_DistanceLeft': 'float64', 'ET_DistanceRight': 'float64', 'ET_GyroX': 'float64', 'ET_GyroY': 'float64', 'ET_GyroZ': 'float64', 'ET_AccX': 'float64', 'ET_AccY': 'float64', 'ET_AccZ': 'float64', 'ET_HeadRotationPitch': 'float64', 'ET_HeadRotationYaw': 'float64', 'ET_HeadRotationRoll': 'float64'}
✅ CLEANING APPLIED CORRECTLY — CELL 8 — Numeric coercion and canonical order


In [10]:
# ============================================================
# CELL 9 — Final Phase 1B validation report
# This cell checks row count, schema, required columns, metadata, and missing-value coverage.
# ============================================================

missing_required_final = sorted([c for c in REQUIRED_MIN if c not in set(df_pruned.columns)])
schema_exact = list(df_pruned.columns) == CANONICAL_ORDER
row_count_same = len(df_pruned) == len(df_raw)
no_extra_cols = set(df_pruned.columns) == set(CANONICAL_ORDER)
metadata_present = all(c in df_pruned.columns for c in META_ONLY)
all_numeric_ok = all(pd.api.types.is_numeric_dtype(df_pruned[c]) for c in df_pruned.columns if c in NUMERIC_LIKE)

print("Schema exact order       :", schema_exact)
print("Row count unchanged     :", row_count_same, "| raw:", len(df_raw), "| final:", len(df_pruned))
print("No extra columns         :", no_extra_cols)
print("Missing required final   :", missing_required_final)
print("Metadata present         :", metadata_present)
print("Numeric-like columns OK  :", all_numeric_ok)
print("Final schema signature   :", schema_sig)

print("\nMetadata unique values after Phase 1B:")
for c in META_ONLY[:3]:
    print(f"{c}:", sorted(pd.Series(df_pruned[c]).dropna().unique())[:20])

print("\nMissing-value percentage by column:")
missing_pct = (df_pruned.isna().mean() * 100).round(2).sort_values(ascending=False)
display(missing_pct.to_frame("missing_%"))

final_passed = (
    schema_exact
    and row_count_same
    and no_extra_cols
    and len(missing_required_final) == 0
    and metadata_present
    and all_numeric_ok
)

print_step_result("FINAL PHASE 1B VALIDATION", final_passed)

Schema exact order       : True
Row count unchanged     : True | raw: 11902 | final: 11902
No extra columns         : True
Missing required final   : []
Metadata present         : True
Numeric-like columns OK  : True
Final schema signature   : b9f372cf

Metadata unique values after Phase 1B:
subject_id: [np.int64(10)]
task: [np.int64(2)]
trial: [np.int64(4)]

Missing-value percentage by column:


,missing_%
ET_GazeLefty,88.88
ET_GazeLeftx,88.88
ET_PupilLeft,88.88
ET_PupilRight,88.88
ET_GazeRighty,88.88
ET_GazeRightx,88.88
ET_HeadRotationRoll,88.88
ET_GyroZ,88.88
ET_AccX,88.88
ET_AccY,88.88


✅ CLEANING APPLIED CORRECTLY — FINAL PHASE 1B VALIDATION


In [11]:
# ============================================================
# CELL 10 — Export Phase 1B cleaned CSV, schema lock, and report
# This cell saves the pruned CSV and verification files for professor review.
# ============================================================

clean_output_path = OUTPUT_DIR / f"{INPUT_CSV.stem}_phase1B_train_schema_checked.csv"
report_output_path = OUTPUT_DIR / "trainschema_prune_report_single.csv"
schema_lock_path = OUTPUT_DIR / "train_schema.json"

df_pruned.to_csv(clean_output_path, index=False, encoding="utf-8-sig")

info = {
    "kept_n": len(df_pruned.columns),
    "dropped_n": len(drop_cols),
    "missing_required": ";".join(missing_required_final),
    "added_missing": ";".join(added_missing),
    "kept_cols": ";".join(df_pruned.columns),
    "dropped_cols": ";".join(sorted(drop_cols)),
    "schema_sig": schema_sig,
}

report_df = pd.DataFrame([{
    "subject": "single_upload",
    "file": INPUT_CSV.name,
    "status": "ok" if len(missing_required_final) == 0 else "ok_with_missing",
    "reason": "" if len(missing_required_final) == 0 else "missing_required",
    **info
}])
report_df.to_csv(report_output_path, index=False, encoding="utf-8-sig")

schema_lock = {
    "meta_only": META_ONLY,
    "eeg_cols": EEG_COLS,
    "emg_raw": EMG_RAW,
    "et_core": ET_CORE,
    "et_dist": ET_DIST,
    "imu_head": IMU_HEAD,
    "canonical_order": CANONICAL_ORDER,
    "required_min": sorted(REQUIRED_MIN),
    "always_drop": sorted(ALWAYS_DROP),
    "defaults": FILL_DEFAULTS,
    "require_emg": REQUIRE_EMG,
}
with schema_lock_path.open("w", encoding="utf-8") as f:
    json.dump(schema_lock, f, indent=2)

print("Saved Phase 1B CSV :", clean_output_path)
print("Saved report       :", report_output_path)
print("Saved schema lock  :", schema_lock_path)

print("\nConfirm saved CSV exists:", clean_output_path.exists())
saved_df = pd.read_csv(clean_output_path)
print("Saved CSV shape:", saved_df.shape)
print("Saved CSV columns match canonical:", list(saved_df.columns) == CANONICAL_ORDER)
display(saved_df.head(5))

passed = clean_output_path.exists() and report_output_path.exists() and schema_lock_path.exists() and list(saved_df.columns) == CANONICAL_ORDER
print_step_result("CELL 10 — Phase 1B files exported", passed)

Saved Phase 1B CSV : phase1B_outputs/063_T24 (1)_phase1B_train_schema_checked.csv
Saved report       : phase1B_outputs/trainschema_prune_report_single.csv
Saved schema lock  : phase1B_outputs/train_schema.json

Confirm saved CSV exists: True
Saved CSV shape: (11902, 38)
Saved CSV columns match canonical: True


,subject_id,task,trial,Timestamp_seconds,Ch1,Ch2,Ch3,Ch4,Ch5,Ch6,Ch7,Ch8,Ch1 EMG raw,Ch2 EMG raw,Ch3 EMG raw,Ch4 EMG raw,ET_GazeLeftx,ET_GazeLefty,ET_GazeRightx,ET_GazeRighty,ET_PupilLeft,ET_PupilRight,ET_ValidityLeftEye,ET_ValidityRightEye,ET_Blink,ET_Fixation,ET_Worn,ET_DistanceLeft,ET_DistanceRight,ET_GyroX,ET_GyroY,ET_GyroZ,ET_AccX,ET_AccY,ET_AccZ,ET_HeadRotationPitch,ET_HeadRotationYaw,ET_HeadRotationRoll
0,10,2,4,0.000000e+00,104424.920,97475.305,104743.68,117912.734,134634.22,61569.080,62151.535,45519.785,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10,2,4,1.499429e-07,104424.920,97475.305,104743.68,117912.734,134634.22,61569.080,62151.535,45519.785,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10,2,4,2.000000e-03,104432.734,97476.400,104738.30,117938.625,134648.58,61575.230,62170.227,45518.310,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10,2,4,2.000150e-03,104432.734,97476.400,104738.30,117938.625,134648.58,61575.230,62170.227,45518.310,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10,2,4,4.000000e-03,104458.960,97479.930,104761.28,117996.180,134697.36,61586.434,62254.438,45564.180,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


✅ CLEANING APPLIED CORRECTLY — CELL 10 — Phase 1B files exported


In [12]:
# ============================================================
# CELL 11 — Optional original-style batch scan over Sub-* folders
# This cell is optional. It scans ROOT_DIR/Sub-*/cleaned/*.csv like your original notebook.
# IMPORTANT: By default, RUN_BATCH_SCAN=False to avoid accidentally overwriting many files.
# ============================================================

RUN_BATCH_SCAN = False

ROOT_DIR = Path("/content/Data")       # Change this if your Sub-* folders are in Google Drive/OneDrive mount
SUBJECT_GLOB = "Sub-*"
TARGET_SUBDIR = "cleaned"
REPORT_NAME = "trainschema_prune_report.csv"
SCHEMA_LOCK = "train_schema.json"

OVERWRITE_ORIGINAL_FILES = False       # False = save copies into OUTPUT_DIR/batch_pruned
DELETE_POLICY = "add_missing"

def should_delete_for_missing(missing_required: list) -> bool:
    if DELETE_POLICY == "keep_bad":
        return False
    if DELETE_POLICY == "add_missing":
        return False
    return bool(missing_required)

def prune_to_train_schema(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    cols_present = set(df.columns)
    keep_set = set(CANONICAL_ORDER)

    missing_canonical = [c for c in CANONICAL_ORDER if c not in cols_present]

    added_missing = []
    if DELETE_POLICY == "add_missing" and missing_canonical:
        df = add_missing_columns(df, missing_canonical)
        added_missing = missing_canonical
        cols_present = set(df.columns)

    missing_required = sorted([c for c in REQUIRED_MIN if c not in cols_present])

    drop_set = (cols_present - keep_set) | (cols_present & ALWAYS_DROP)
    pruned = df.drop(columns=[c for c in drop_set if c in df.columns], errors="ignore")

    pruned = coerce_types(pruned)
    pruned = pruned.reindex(columns=CANONICAL_ORDER, fill_value=np.nan)

    schema_sig = hashlib.md5(("|".join(pruned.columns)).encode()).hexdigest()[:8]

    info = {
        "kept_n": len(pruned.columns),
        "dropped_n": len(drop_set),
        "missing_required": ";".join(missing_required),
        "added_missing": ";".join(added_missing),
        "kept_cols": ";".join(pruned.columns),
        "dropped_cols": ";".join(sorted(drop_set)),
        "schema_sig": schema_sig,
    }
    return pruned, info

def write_schema_lock(root_dir: Path):
    lock = {
        "meta_only": META_ONLY,
        "eeg_cols": EEG_COLS,
        "emg_raw": EMG_RAW,
        "et_core": ET_CORE,
        "et_dist": ET_DIST,
        "imu_head": IMU_HEAD,
        "canonical_order": CANONICAL_ORDER,
        "required_min": sorted(REQUIRED_MIN),
        "always_drop": sorted(ALWAYS_DROP),
        "defaults": FILL_DEFAULTS,
        "require_emg": REQUIRE_EMG,
    }
    out = root_dir / SCHEMA_LOCK
    with out.open("w", encoding="utf-8") as f:
        json.dump(lock, f, indent=2)
    return out

def verify_prune_save_batch():
    rows = []
    total = good = bad = 0
    schema_sigs = set()

    print(f"ROOT_DIR = {ROOT_DIR}")
    print(f"DELETE_POLICY = {DELETE_POLICY} | REQUIRE_EMG = {REQUIRE_EMG}")
    print(f"OVERWRITE_ORIGINAL_FILES = {OVERWRITE_ORIGINAL_FILES}")

    if OVERWRITE_ORIGINAL_FILES:
        lock_path = write_schema_lock(ROOT_DIR)
    else:
        batch_out = OUTPUT_DIR / "batch_pruned"
        batch_out.mkdir(parents=True, exist_ok=True)
        lock_path = write_schema_lock(batch_out)

    print(f"[lock] schema → {lock_path}")

    for subj_dir in sorted([p for p in ROOT_DIR.glob(SUBJECT_GLOB) if p.is_dir()]):
        scan_dir = subj_dir / TARGET_SUBDIR
        if not scan_dir.exists():
            continue

        csvs = sorted(scan_dir.glob("*.csv"))
        if not csvs:
            continue

        print(f"\n=== Pruning {subj_dir.name}/{TARGET_SUBDIR}: {len(csvs)} files ===")

        for csv_path in csvs:
            total += 1
            df = read_csv_safely(csv_path)

            if df is None:
                print(f"[READ-FAIL] {csv_path.name} (kept untouched)")
                rows.append({
                    "subject": subj_dir.name, "file": csv_path.name,
                    "status": "read_failed_kept",
                    "reason": "read_failed",
                    "missing_required": "", "added_missing": "",
                    "kept_n": "", "dropped_n": "", "kept_cols": "", "dropped_cols": "",
                    "schema_sig": ""
                })
                bad += 1
                continue

            pruned, info = prune_to_train_schema(df)
            missing_req = info["missing_required"].split(";") if info["missing_required"] else []

            if should_delete_for_missing(missing_req):
                if OVERWRITE_ORIGINAL_FILES:
                    try:
                        csv_path.unlink()
                        print(f"[DELETED] {csv_path.name} (missing required: {missing_req})")
                        status = "deleted_missing_required"
                    except Exception as e:
                        print(f"[WARN] Could not delete {csv_path.name}: {e}")
                        status = "delete_failed"
                else:
                    print(f"[SKIPPED-BAD] {csv_path.name} missing required: {missing_req}")
                    status = "skipped_missing_required"
                bad += 1
            else:
                if OVERWRITE_ORIGINAL_FILES:
                    save_path = csv_path
                else:
                    rel = csv_path.relative_to(ROOT_DIR)
                    save_path = OUTPUT_DIR / "batch_pruned" / rel
                    save_path.parent.mkdir(parents=True, exist_ok=True)

                pruned.to_csv(save_path, index=False, encoding="utf-8-sig")
                add_msg = f", +{info['added_missing']}" if info["added_missing"] else ""
                print(f"[SAVED] {csv_path.name}: kept {info['kept_n']} cols, dropped {info['dropped_n']}{add_msg} | schema {info['schema_sig']}")
                status = "ok" if not missing_req else "ok_with_missing"
                good += 1
                schema_sigs.add(info["schema_sig"])

            rows.append({
                "subject": subj_dir.name,
                "file": csv_path.name,
                "status": status,
                "reason": "" if status.startswith("ok") else ("missing_required" if missing_req else ""),
                **info
            })

    report_root = ROOT_DIR if OVERWRITE_ORIGINAL_FILES else (OUTPUT_DIR / "batch_pruned")
    report_path = report_root / REPORT_NAME
    pd.DataFrame(rows).to_csv(report_path, index=False, encoding="utf-8-sig")

    print(f"\n=== Batch Summary ===")
    print(f"Total: {total} | Good: {good} | Bad: {bad}")
    print(f"Report saved: {report_path}")
    if len(schema_sigs) == 1:
        print("[schema] All files share ONE canonical schema.")
        batch_passed = True
    else:
        print(f"[schema] WARNING: {len(schema_sigs)} different schema signatures encountered.")
        batch_passed = False

    print_step_result("CELL 11 — Batch schema pruning", total > 0 and good > 0 and batch_passed)

if RUN_BATCH_SCAN:
    verify_prune_save_batch()
else:
    print("Batch scan skipped. Set RUN_BATCH_SCAN=True only when your ROOT_DIR is correct.")

Batch scan skipped. Set RUN_BATCH_SCAN=True only when your ROOT_DIR is correct.
